# Session 1: Data Preprocessing

Suppose we want an AI model to predict whether a student will pass a course. To do that we need to collect a **dataset**: a table of observations.

- Each row is one observation, also called a **sample**.
- Each input column is a **feature**: age, city, study hours, attendance.
- The answer we want to predict is the **target**: result.


| Student ID | Age | City       | Attendance (%) | Study Hours | Passed |
|-----------:|----:|------------|---------------:|------------:|--------|
| 101        | 20  | Cairo      | 92             | 3.5         | Yes    |
| 102        | NaN | Giza       | 75             | 1.0         | No     |
| 103        | 21  | cairo      | 88             | 4.0         | Yes    |
| 103        | 21  | cairo      | 88             | 4.0         | Yes    |
| 104        | 19  | Alexandria | 60             | 0.5         | No     |

But before using the dataset to train our model, we need to clean the data. For example, the table above has one age missing, Cairo and cairo represent the same city but are different strings, city names are text (most models expect numerical inputs), and attendance and study hours use different numerical scales (one will cause higher impact on the result than the other when it shouldn't be the case). **Preprocessing** means inspecting, correcting, and transforming raw data into a suitable representation for a model.




## Steps to Preprocess Data

### 1. Explore
Exploration tells us what needs to be fixed, such as missing values, but doesn't touch anything yet. We use a python library called `pandas` to help us read csv files as `DataFrame` objects (table-like objects) which are easy to explore. A `Series` is an object that represents one column in a `DataFrame`.

**Basic DataFrame Operations:**
| Code | Inside the brackets | What it does | Returns |
|---|---|---|---|
| `df['age']` | One column name | Selects the `age` column | A **Series** |
| `df[['age', 'hours']]` | A list of column names | Selects those columns in that order | A **DataFrame** |
| `df[['age']]` | A list containing one column name | Selects `age` while keeping the table structure | A **DataFrame** |
| `df[row_mask]` | A Boolean Series | Keeps rows marked `True`; excludes rows marked `False` | A **DataFrame**, possibly with fewer rows |
| `df[cell_mask]` | A Boolean DataFrame | Keeps cells marked `True`; replaces cells marked `False` with missing values | A **DataFrame** with the same shape when labels match |
| `df[0:2]` | An integer slice | Selects rows at positions 0 and 1; position 2 is excluded | A **DataFrame** |

In [ ]:
import pandas as pd

df = pd.read_csv('student_data_dirty.csv') #convert csv file to DataFrame object

print(f"Shape: {df.shape}") #returns a tuple (number of rows, number of columns)
print("\nInfo:")
df.info() #prints info like column names, data types, etc.
print(f"\nDescribe:\n {df.describe()}") #prints a DataFrame the represents statistical summury of df such as each column's count, mean, min, max, etc.
print(f"\nMissing Value Counts:\n {df.isna().sum()}\n") #df.isna() returns a same size boolean DataFrame with True on every cell with a missing value, .sum() sums the True values in each column, the result is a Series with the missing-value counts for each corresponding column in df
print(f"\nDuplicated Rows Counts:\n {df.duplicated().sum()}") #df.duplicated() returns boolean Series with True for every duplicated row, the first occurrence is False by default. .sum() sums the True values.
print(f"\nFrequencies:\n {df['result'].value_counts()}") #returns a Series mapping each distinct value in the results column to its frequency

Shape: (83, 11)

Info:
<class 'pandas.DataFrame'>
RangeIndex: 83 entries, 0 to 82
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   student_id          83 non-null     str    
 1   age                 77 non-null     float64
 2   city                79 non-null     str    
 3   study_hours         78 non-null     str    
 4   attendance          83 non-null     int64  
 5   family_income       79 non-null     str    
 6   education_level     80 non-null     str    
 7   enrollment_date     83 non-null     str    
 8   review_text         83 non-null     str    
 9   result              83 non-null     str    
 10  mostly_empty_notes  3 non-null      str    
dtypes: float64(1), int64(1), str(9)
memory usage: 7.3 KB

Describe:
               age  attendance
count   77.000000   83.000000
mean    24.142857   65.759036
std     26.288738   21.337354
min     -3.000000   35.000000
25%     20.000000   46.00000

### 2. Clean

In [6]:
df.drop_duplicates() #drops duplicated rows and keeps the first occurance
df.dropna(axis=1, thresh=len(df) // 2) #drops columns (axis = 1) with missing values exceeding thresh (in this case half the number of rows)
df['enrollment_date'] = pd.to_datetime(df['enrollment_date'], errors='coerce') #converts dates columns into datetime values, errors='coerce' makes invalid values NaT (like null)

After dropping duplicated rows and columns with a lot of missing values and converting dates to datetime values, we now want fill missing values. We use a python library called `scikit-learn` for that. We could fill missing values with either
1. mean of their corresponding column
2. most frequent value of their corresponding column
3. median of their corresponding column
4. a constant value you choose

In this example we use median.

In [8]:
from sklearn.impute import SimpleImputer

md_imp = SimpleImputer(strategy='median') #creates an imputer object
df[['age', 'attendance']] = md_imp.fit_transform(df[['age', 'attendance']])

**Notes:** 
- `fit_transform()` does two things: `fit` caclulates the median (or mean, most frequent value, etc.) and `transform` fills the missing values with it. Originally the function was two different functions.
- `df[['column name 1', 'column name 2', ...]]` returns a DataFrame with the columns selected. `fit_transform()` expects a DataFrame, therefore we use `df[['column name']]` even if we'll only transform one column. The result is then a `NumPy` array (a 2d array) which could then be written back inside the actual df.